# 🤗 Sentiment Analysis with DistilBERT: Full Fine-Tuning on SST-2

**Author:** Sonal Mishra  
**Dataset:** [SST-2 (Stanford Sentiment Treebank)](https://huggingface.co/datasets/stanfordnlp/sst2)  
**Model:** `distilbert-base-uncased` → Fine-tuned for Binary Sentiment Classification  
**Platform:** Google Colab (T4 GPU)

---

## 📌 What This Notebook Covers

This notebook demonstrates **full fine-tuning** of a pre-trained DistilBERT model on binary sentiment classification (positive/negative).

> **Analogy:** Think of DistilBERT as a well-read graduate who has absorbed vast general knowledge (pre-training).  
> Fine-tuning is like enrolling them in a specialized course — they don't start from scratch, they just sharpen their skills for a specific task.

### Pipeline at a Glance
```
Pre-trained DistilBERT
        ↓
Load SST-2 Dataset (67K movie reviews)
        ↓
Tokenize → Pad → Truncate
        ↓
Full Fine-Tuning (3 epochs, lr=2e-5)
        ↓
Evaluation + Inference
```

### 📊 Results Summary
| Stage | Accuracy |
|-------|----------|
| Base Model (no fine-tuning) | 50.5% |
| After Fine-Tuning (3 epochs) | 91.5% |
| **Improvement** | **+41.0%** |


---
## Section 1: Install Dependencies

We use four core HuggingFace libraries:
- `transformers` — pre-trained models and tokenizers
- `datasets` — easy dataset loading from HuggingFace Hub
- `evaluate` — standardized metrics (accuracy, F1, etc.)
- `accelerate` — backend for distributed/GPU training via the Trainer API


In [1]:
# Install HuggingFace ecosystem libraries
# -q flag suppresses verbose pip output (cleaner in Colab)
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00


---
## Section 2: Load Pre-trained DistilBERT Model & Tokenizer

**DistilBERT** is a lighter, faster version of BERT — 40% smaller, 60% faster, retaining 97% of BERT's language understanding.

> **Analogy:** If BERT is a full encyclopedia, DistilBERT is a condensed study guide  
> — same essential knowledge, but optimized for speed.

### Architecture Recap
- **6 Transformer layers** (BERT-base has 12)
- **768 hidden dimensions**
- **12 attention heads**
- **66M parameters** total

We load `distilbert-base-uncased` (pre-trained on English Wikipedia + BookCorpus) and add a **classification head** (`num_labels=2`) on top.  
The two new layers (`pre_classifier` + `classifier`) are randomly initialized — they will learn during fine-tuning.


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_NAME = "distilbert-base-uncased"   # Lightweight BERT variant
NUM_LABELS = 2                            # Binary: 0 = Negative, 1 = Positive

# ── Load Tokenizer ──────────────────────────────────────────────────────────
# The tokenizer converts raw text → token IDs that the model understands
# AutoTokenizer automatically selects the correct tokenizer for the model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ── Load Model ──────────────────────────────────────────────────────────────
# AutoModelForSequenceClassification adds a linear classification head
# on top of the base DistilBERT encoder
# Note: pre_classifier and classifier weights are randomly initialized
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

print("✅ Model loaded successfully!")
print(model)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded successfully!
DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): D

---
## Section 3: Load the SST-2 Dataset

**SST-2** (Stanford Sentiment Treebank v2) is a widely used benchmark for binary sentiment classification.  
Each sample is a movie review sentence labeled as **positive (1)** or **negative (0)**.

### Dataset Stats
| Split | Samples |
|-------|---------|
| Train | 67,349 |
| Validation | 872 |
| Test | 1,821 |


In [3]:
from datasets import load_dataset

# ── Load SST-2 from HuggingFace Hub ─────────────────────────────────────────
# Returns a DatasetDict with train / validation / test splits
dataset = load_dataset("sst2")

print("📦 Dataset Overview:")
print(dataset)

print("\n🔍 Sample Training Example:")
print(dataset["train"][0])



README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

📦 Dataset Overview:
DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

🔍 Sample Training Example:
{'idx': 0, 'sentence': 'hide new secretions from the parental units ', 'label': 0}


---
## Section 4: Evaluate Baseline Accuracy (Before Fine-Tuning)

Before any training, we test how well the raw pre-trained DistilBERT performs on sentiment classification.

Since the classification head is randomly initialized, we expect near **random performance (~50%)**.

> **Analogy:** Asking a general-knowledge expert to predict stock prices without any finance training — their answer is essentially a coin flip.


In [4]:
import numpy as np

def evaluate_accuracy(model, dataset_split, tokenizer, n_samples=200, device="cpu"):
    """
    Evaluate model accuracy on the first n_samples of a dataset split.

    Args:
        model: HuggingFace sequence classification model
        dataset_split: HuggingFace Dataset split (train/val/test)
        tokenizer: corresponding tokenizer
        n_samples (int): number of samples to evaluate on
        device (str): 'cpu' or 'cuda'

    Returns:
        float: accuracy percentage (0-100)
    """
    model.eval()   # Disable dropout and batch norm layers during inference
    model.to(device)

    correct = 0
    total = 0

    for i in range(n_samples):
        row = dataset_split[i]

        # Tokenize: convert sentence to tensor of token IDs
        inputs = tokenizer(
            row["sentence"],
            return_tensors="pt",    # Return PyTorch tensors
            truncation=True,         # Trim sequences longer than max_length
            max_length=512           # DistilBERT's max sequence length
        ).to(device)

        # Inference: no gradient computation needed (saves memory + time)
        with torch.no_grad():
            outputs = model(**inputs)
            # outputs.logits shape: [1, 2] — raw scores for each class
            prediction = torch.argmax(outputs.logits, dim=-1).item()

        if prediction == row["label"]:
            correct += 1
        total += 1

    return (correct / total) * 100


# ── Determine device ──────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {device}")

# ── Run Baseline Evaluation ──────────────────────────────────────────────────
print("\n=== BASELINE ACCURACY (Before Fine-Tuning) ===")
baseline_accuracy = evaluate_accuracy(
    model, dataset["validation"], tokenizer, n_samples=200, device=device
)
print(f"Accuracy: {baseline_accuracy:.1f}%")
print("→ Expected ~50% since the classification head is randomly initialized")


🖥️  Using device: cuda

=== BASELINE ACCURACY (Before Fine-Tuning) ===
Accuracy: 49.5%
→ Expected ~50% since the classification head is randomly initialized


---
## Section 5: Tokenize the Dataset

Before training, we convert raw text into token IDs that the model can process.

### What Tokenization Does
1. **Splits** text into subword tokens (e.g., "playing" → ["play", "##ing"])
2. **Maps** tokens to integer IDs from the vocabulary (30,522 tokens in BERT vocab)
3. **Adds** special tokens: `[CLS]` at start, `[SEP]` at end
4. **Pads** sequences to a uniform length (128 tokens here)
5. **Truncates** sequences longer than max_length

> **Analogy:** Tokenization is like converting a sentence into a barcode — the model doesn't read words, it reads numbers.

**Why `max_length=128`?**  
SST-2 sentences are short movie review snippets. 128 tokens covers 99%+ of them.  
Using 512 would triple training time with no accuracy benefit.


In [5]:
from transformers import TrainingArguments, Trainer
import evaluate

# ── Tokenization Function ────────────────────────────────────────────────────
def tokenize_function(batch):
    """
    Tokenize a batch of sentences.
    Called with batched=True, so 'batch' is a dict of lists.

    Returns:
        dict with input_ids, attention_mask (and token_type_ids if applicable)
    """
    return tokenizer(
        batch["sentence"],
        truncation=True,           # Trim to max_length if sentence is too long
        max_length=128,            # SST-2 reviews are short; 128 is sufficient
        padding="max_length"       # Pad shorter sequences to uniform 128 length
    )

# ── Apply tokenization to all splits ────────────────────────────────────────
# batched=True speeds up processing by tokenizing in chunks instead of row-by-row
print("🔤 Tokenizing train split...")
tokenized_train = dataset["train"].map(tokenize_function, batched=True)

print("🔤 Tokenizing validation split...")
tokenized_val = dataset["validation"].map(tokenize_function, batched=True)

print("\n✅ Tokenization complete!")
print(f"Train tokens shape example: {tokenized_train[0]['input_ids'][:10]}...")


🔤 Tokenizing train split...


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

🔤 Tokenizing validation split...


Map:   0%|          | 0/872 [00:00<?, ? examples/s]


✅ Tokenization complete!
Train tokens shape example: [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102]...


---
## Section 6: Fine-Tune with HuggingFace Trainer

The HuggingFace `Trainer` API handles the full training loop:  
gradient computation, optimizer steps, learning rate scheduling, checkpointing, and evaluation.

### Key Hyperparameters Explained

| Hyperparameter | Value | Why |
|---|---|---|
| `learning_rate` | 2e-5 | Standard for fine-tuning BERT-family models — large LR destroys pre-trained weights |
| `num_train_epochs` | 3 | Enough to converge on SST-2; more risks overfitting |
| `batch_size` | 32 | Balances GPU memory usage and training stability |
| `eval_strategy` | epoch | Evaluate after every full pass over training data |
| `load_best_model_at_end` | True | Automatically restore the checkpoint with best validation accuracy |

> **Analogy:** `load_best_model_at_end=True` is like a student who saves their best exam answer sheet  
> and submits that — even if their last attempt was worse than their second attempt.


In [6]:
# ── Metric Setup ─────────────────────────────────────────────────────────────
# Load accuracy metric from the HuggingFace evaluate library
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """
    Called by Trainer after each evaluation step.

    Args:
        eval_pred: tuple of (raw_predictions, true_labels)
                   raw_predictions shape: [n_samples, num_labels]

    Returns:
        dict with 'accuracy' key
    """
    raw_predictions, labels = eval_pred

    # Convert raw logits → predicted class index
    predictions = np.argmax(raw_predictions, axis=1)

    return accuracy_metric.compute(predictions=predictions, references=labels)


# ── Training Configuration ───────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./distilbert-sentiment",      # Checkpoints saved here
    num_train_epochs=3,                        # 3 full passes over training data
    per_device_train_batch_size=32,            # Training batch size
    per_device_eval_batch_size=32,             # Evaluation batch size
    eval_strategy="epoch",                     # Run eval after every epoch
    save_strategy="epoch",                     # Save checkpoint every epoch
    learning_rate=2e-5,                        # Conservative LR for fine-tuning
    load_best_model_at_end=True,               # Restore best checkpoint at end
    logging_steps=50,                          # Log training loss every 50 steps
    report_to="none",                          # Disable W&B / MLflow logging
)

# ── Initialize Trainer ───────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# ── Start Training ───────────────────────────────────────────────────────────
print("🚀 Starting fine-tuning...")
print(f"   Total training samples : {len(tokenized_train)}")
print(f"   Batch size             : 32")
print(f"   Steps per epoch        : {len(tokenized_train) // 32}")
print(f"   Total epochs           : 3\n")

trainer.train()


🚀 Starting fine-tuning...
   Total training samples : 67349
   Batch size             : 32
   Steps per epoch        : 2104
   Total epochs           : 3



Epoch,Training Loss,Validation Loss,Accuracy
1,0.161868,0.268182,0.899083
2,0.108848,0.312431,0.901376
3,0.092519,0.371662,0.899083


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=6315, training_loss=0.14105967291743626, metrics={'train_runtime': 2133.5187, 'train_samples_per_second': 94.701, 'train_steps_per_second': 2.96, 'total_flos': 6691160124062208.0, 'train_loss': 0.14105967291743626, 'epoch': 3.0})

---
## Section 7: Save the Fine-Tuned Model

We save both the model weights and the tokenizer so they can be loaded together later —  
either locally or pushed to HuggingFace Hub.


In [7]:
SAVE_PATH = "./distilbert-sentiment"

# ── Save model weights and config ────────────────────────────────────────────
model.save_pretrained(SAVE_PATH)

# ── Save tokenizer files (vocab, config, special tokens) ─────────────────────
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ Model and tokenizer saved to: {SAVE_PATH}")
print("\nFiles saved:")
import os
for f in os.listdir(SAVE_PATH):
    print(f"  📄 {f}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model and tokenizer saved to: ./distilbert-sentiment

Files saved:
  📄 model.safetensors
  📄 config.json
  📄 tokenizer_config.json
  📄 tokenizer.json
  📄 checkpoint-4210
  📄 checkpoint-6315
  📄 checkpoint-2105


---
## Section 8: Evaluate Fine-Tuned Model Accuracy

We re-run the same evaluation function on the validation set to measure the improvement after fine-tuning.


In [8]:
# ── Re-evaluate after fine-tuning ────────────────────────────────────────────
print("=== FINE-TUNED MODEL ACCURACY ===")
finetuned_accuracy = evaluate_accuracy(
    model, dataset["validation"], tokenizer, n_samples=200, device=device
)
print(f"Accuracy: {finetuned_accuracy:.1f}%")


=== FINE-TUNED MODEL ACCURACY ===
Accuracy: 91.5%


---
## Section 9: Real-World Inference — Predict Sentiment on Custom Sentences

Let's test our fine-tuned model on custom movie review sentences to see it in action.


In [9]:
def predict_sentiment(sentence, model, tokenizer, device="cpu"):
    """
    Predict the sentiment of a single input sentence.

    Args:
        sentence (str): Raw input text
        model: Fine-tuned DistilBERT classification model
        tokenizer: Corresponding tokenizer
        device (str): 'cpu' or 'cuda'

    Returns:
        str: 'POSITIVE ✅' or 'NEGATIVE ❌'
    """
    model.eval()

    # Tokenize and send to device
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    # Run inference without gradient tracking
    with torch.no_grad():
        outputs = model(**inputs)
        # Pick the class with the highest score
        prediction = torch.argmax(outputs.logits, dim=-1).item()

    return "POSITIVE ✅" if prediction == 1 else "NEGATIVE ❌"


# ── Test Sentences ────────────────────────────────────────────────────────────
test_sentences = [
    "This movie was absolutely brilliant and touching",
    "Terrible film, complete waste of time",
    "The acting was not bad but the plot was boring",
    "One of the best movies I have ever seen",
    "I would not recommend this to anyone",
]

# ── Run Predictions ───────────────────────────────────────────────────────────
print("=== SENTIMENT PREDICTIONS ===\n")
for sentence in test_sentences:
    result = predict_sentiment(sentence, model, tokenizer, device=device)
    print(f"{result} | {sentence}")


=== SENTIMENT PREDICTIONS ===

POSITIVE ✅ | This movie was absolutely brilliant and touching
NEGATIVE ❌ | Terrible film, complete waste of time
NEGATIVE ❌ | The acting was not bad but the plot was boring
POSITIVE ✅ | One of the best movies I have ever seen
NEGATIVE ❌ | I would not recommend this to anyone


---
## Section 10: Final Results Summary


In [10]:
# ── Print Before vs After Comparison ─────────────────────────────────────────
print("=" * 50)
print("         FINAL RESULTS SUMMARY")
print("=" * 50)
print(f"  Base model accuracy    : {baseline_accuracy:.1f}%   (untrained head)")
print(f"  Fine-tuned accuracy    : {finetuned_accuracy:.1f}%   (3 epochs, SST-2)")
print(f"  Improvement            : +{finetuned_accuracy - baseline_accuracy:.1f}%")
print("=" * 50)
print()
print("📌 Key Takeaways:")
print("  • DistilBERT's pre-trained language understanding transfers directly")
print("    to downstream classification tasks.")
print("  • Only the classification head starts from scratch — the encoder")
print("    already knows language; it just needs task-specific calibration.")
print("  • 3 epochs on 67K samples was sufficient to achieve 91.5% accuracy.")


         FINAL RESULTS SUMMARY
  Base model accuracy    : 49.5%   (untrained head)
  Fine-tuned accuracy    : 91.5%   (3 epochs, SST-2)
  Improvement            : +42.0%

📌 Key Takeaways:
  • DistilBERT's pre-trained language understanding transfers directly
    to downstream classification tasks.
  • Only the classification head starts from scratch — the encoder
    already knows language; it just needs task-specific calibration.
  • 3 epochs on 67K samples was sufficient to achieve 91.5% accuracy.


---
## 📚 Key Concepts Recap

| Concept | Explanation |
|---|---|
| **Full Fine-Tuning** | All model layers are updated during training (vs. LoRA / adapters which freeze most layers) |
| **Transfer Learning** | Leveraging pre-trained weights instead of training from random initialization |
| **Tokenization** | Converting text → subword token IDs the model can process |
| **Attention Mask** | Tells the model which positions are real tokens (1) vs. padding (0) |
| `load_best_model_at_end` | Automatically restores the checkpoint with best validation score |

---

## 🔗 References
- [DistilBERT Paper](https://arxiv.org/abs/1910.01108)
- [HuggingFace Transformers Docs](https://huggingface.co/docs/transformers)
- [SST-2 Dataset](https://huggingface.co/datasets/stanfordnlp/sst2)

---
*Built with ❤️ using HuggingFace Transformers | Google Colab T4 GPU*
